In [1]:
import re, unicodedata
from rank_bm25 import BM25Okapi   # pip install rank-bm25

STOPWORDS = {"le","la","les","de","des","du","un","une","et","ou","en","à","a",
             "au","aux","pour","par","sur","dans","est","que","qui","ce","se","l","d"}

def tokeniser(texte: str) -> list[str]:
    texte = unicodedata.normalize("NFD", texte.lower())
    texte = "".join(c for c in texte if unicodedata.category(c) != "Mn")  # enlève les accents
    return [t for t in re.findall(r"\w+", texte) if t not in STOPWORDS]

In [2]:
"degats"==tokeniser("Dégâts")

False

In [3]:
tokeniser("Dégâts")

['degats']

In [5]:
# version 2
from nltk.stem.snowball import FrenchStemmer     # pip install nltk (pas de téléchargement nécessaire)
stem = FrenchStemmer().stem

STOPWORDS |= {"suis", "me", "moi", "mon", "etre", "ai", "est-ce"}

def tokeniser(texte):
    texte = texte.lower().replace("ʼ", "'").replace("’", "'")
    texte = unicodedata.normalize("NFD", texte)
    texte = "".join(c for c in texte if unicodedata.category(c) != "Mn")
    return [stem(t) for t in re.findall(r"\w+", texte) if len(t) > 1 and t not in STOPWORDS]

# print(tokeniser(question))                     # attendu : 'vol', 'cav'... ("vole" et "vol" -> même racine)
# bm25 = BM25Okapi([tokeniser(t) for t in data["documents"]])   # on reconstruit l'index avec le nouveau tokeniseur

In [6]:
def construire_index_bm25(chunks):          # chunks = liste de dicts {id, texte, contrat_id, page...}
    index = {}
    for cid in {c["contrat_id"] for c in chunks}:
        docs = [c for c in chunks if c["contrat_id"] == cid]
        index[cid] = (BM25Okapi([tokeniser(c["texte"]) for c in docs]), docs)
    return index

def rechercher_bm25(index, question, contrat_id, k=20):
    bm25, docs = index[contrat_id]
    scores = bm25.get_scores(tokeniser(question))
    meilleurs = sorted(range(len(docs)), key=lambda i: scores[i], reverse=True)[:k]
    return [docs[i] for i in meilleurs]

In [ ]:
import sys, os
from pathlib import Path

RACINE = Path.cwd().parents[1]          # src/hybride -> racine du projet
os.chdir(RACINE); sys.path.insert(0, str(RACINE))

from src.retrieval.retrieval import ouvrir_store, rechercher
from src.extraction_cp.extraction import analyser_cp

store = ouvrir_store()
_, _, contrat_id = analyser_cp(Path("data/cp/cp_01_luko_etendue.pdf"))
print("Contrat :", contrat_id)

Contrat : luko_hce_2504


In [8]:
data = store.get(where={"contrat_id": contrat_id}, include=["documents", "metadatas"])
print(len(data["documents"]), "chunks pour ce contrat")
print(data["metadatas"][0])
print(data["documents"][0][:300])

359 chunks pour ce contrat
{'reference': 'HCE', 'contrat_id': 'luko_hce_2504', 'produit': 'Habitation', 'type_doc': 'CG', 'fichier': 'luko_habitation-etendue_cg_2504.pdf', 'chunk_id': 'luko_habitation-etendue_cg_2504.pdf#0', 'alias': 'Luko by Allianz Direct, Allianz Direct', 'page': 2, 'edition': '2504', 'assureur': 'Luko', 'formule': 'Couverture Étendue'}
Note à nos lecteurs 

On a voulu vous écrire le plus beau contrat 
d'assurance habitation du monde. 

Avant d'aller dans le détail de ces conditions générales, répondons d'abord à la question à 1 million 
d'euros : à quoi ça sert ? d'où ça sort exactement ?


In [9]:
print(tokeniser("Suis-je couvert si on me vole mon vélo dans ma cave ?"))
print(tokeniser("Les dégâts des eaux et l’assurance"))

['je', 'couvert', 'si', 'on', 'vol', 'velo', 'ma', 'cav']
['degat', 'eau', 'assur']


In [ ]:
from rank_bm25 import BM25Okapi     # pip install rank-bm25

bm25 = BM25Okapi([tokeniser(t) for t in data["documents"]])

question = "Dans quel délai dois-je déclarer un vol à mon assureur ?"
scores = bm25.get_scores(tokeniser(question))
top_bm25 = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]

print("=== BM25")
for i in top_bm25:
    m = data["metadatas"][i]
    print(f"{scores[i]:5.1f}  {m['fichier']} p.{m['page']}  {data['documents'][i][:90]!r}")

print("\n=== Vecteurs")
for p in rechercher(store, question, contrat_id, k=5):
    print(f"       {p.metadata['fichier']} p.{p.metadata['page']}  {p.page_content[:90]!r}")

=== BM25
 15.9  luko_habitation-etendue_cg_2504.pdf p.3  'Une assurance habitation ne vous couvre pas en revanche quand vos biens sont usés à \ncause'
  7.0  luko_habitation-etendue_cg_2504.pdf p.26  "Biens couverts Plafonds d'indemnisation \nHabitation À concurrence des dommages \nContenu Da"
  6.5  luko_habitation-etendue_cg_2504.pdf p.2  'Dans ce cadre, nous faisons attention à toujours employer le mot exact et à prendre le tem'
  6.1  luko_habitation-etendue_cg_2504.pdf p.40  "3.3. Frais de déplacement du mobilier \nLes frais rendus indispensables à la suite d'un sin"
  6.1  luko_habitation-etendue_cg_2504.pdf p.25  "En cas d'inhabitation de vos locaux, la garantie est suspendue dans les conditions suivant"

=== Vecteurs
       luko_habitation-etendue_cg_2504.pdf p.47  'Nous vous garantissons également contre les recours dirigés contre vous en \nraison des dom'
       luko_habitation-etendue_cg_2504.pdf p.12  '3. Quels sont les biens assurés ?'
       luko_habitation-etendue_cg_250

In [11]:
vocab = [set(tokeniser(t)) for t in data["documents"]]
for mot in tokeniser(question):
    print(f"{mot:10} présent dans {sum(mot in v for v in vocab):4} chunks")

suis       présent dans    1 chunks
je         présent dans    5 chunks
couvert    présent dans   20 chunks
si         présent dans  109 chunks
on         présent dans   13 chunks
me         présent dans    0 chunks
vole       présent dans    0 chunks
mon        présent dans    2 chunks
velo       présent dans    0 chunks
ma         présent dans    1 chunks
cave       présent dans    1 chunks


In [11]:
q = tokeniser(question)
print("Tokens de la question :", q, "\n")

def detail(i):
    """Quels mots de la question ce chunk contient, et combien de chunks contiennent chaque mot."""
    toks = tokeniser(data["documents"][i])
    m = data["metadatas"][i]
    print(f"--- p.{m['page']}  score={scores[i]:.1f}")
    for mot in q:
        print(f"   {mot:10} x{toks.count(mot):2} dans ce chunk | présent dans {sum(mot in v for v in vocab):4} chunks")

scores = bm25.get_scores(q)
vocab = [set(tokeniser(t)) for t in data["documents"]]

detail(top_bm25_new := sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[0])   # le 1er de BM25 (p.3)
for i, m in enumerate(data["metadatas"]):                                                     # les bonnes pages
    if m["page"] in (23, 36) and "cg" in m["fichier"]:
        detail(i)

Tokens de la question : ['je', 'couvert', 'si', 'on', 'vol', 'velo', 'ma', 'cav'] 

--- p.3  score=15.9
   je         x 1 dans ce chunk | présent dans    5 chunks
   couvert    x 3 dans ce chunk | présent dans   43 chunks
   si         x 1 dans ce chunk | présent dans  109 chunks
   on         x 1 dans ce chunk | présent dans   13 chunks
   vol        x 0 dans ce chunk | présent dans   26 chunks
   velo       x 0 dans ce chunk | présent dans    0 chunks
   ma         x 2 dans ce chunk | présent dans    1 chunks
   cav        x 0 dans ce chunk | présent dans    4 chunks
--- p.23  score=6.0
   je         x 0 dans ce chunk | présent dans    5 chunks
   couvert    x 1 dans ce chunk | présent dans   43 chunks
   si         x 0 dans ce chunk | présent dans  109 chunks
   on         x 0 dans ce chunk | présent dans   13 chunks
   vol        x 4 dans ce chunk | présent dans   26 chunks
   velo       x 0 dans ce chunk | présent dans    0 chunks
   ma         x 0 dans ce chunk | présent dans    